# Sentinel Gate - Stage 7b: point-in-time candidates (labels 6 weeks late) + score export

**Set `POLICIES` and `PART` in cell 1.** The experiment needs 18 candidates (6 retrain weeks T = 52, 54, ..., 62 × 3 retraining policies). That is too much for one Kaggle session, so run this notebook **three times**, one retraining policy per run:

| Run | `POLICIES` | `PART` | Rough time (4 CPUs) |
|---|---|---|---|
| 1 | `["last_n_weeks"]` | `"last_n_weeks"` | ~3 h |
| 2 | `["all_history"]` | `"all_history"` | ~5-6 h |
| 3 | `["recency_weighted"]` | `"recency_weighted"` | ~5-6 h |

**What this does**
1. Loads the code bundle and installs the *locked* dependencies (same `uv.lock` as the local sandbox).
2. Re-downloads the official EMBER2024 Win64 feature records (SHA-256 verified), rebuilds the Stage 1 splits and checks them against the committed manifest (`docs/cleaning/Win64.json`). Any difference stops the run.
3. Loads v1 from the uploaded `sentinel_v1_Win64.zip` (it is never retrained).
4. Retrains one candidate per retrain week T for the chosen policy, **using only weeks <= T-6** (owner decision 2026-11-24: labels arrive 6 weeks late): fit, threshold and calibration all come from weeks <= T-6.
5. Scores v1 and every candidate on weeks 45-64 and on the never-break set, and counts v1's top-20 features per week (for drift). **Only scores, labels, weeks, label times and file hashes are exported, never feature vectors.**
6. Writes `sentinel_stage7b_Win64_<PART>.zip` to `/kaggle/working` (= the notebook's Output).

**Inputs to attach (Add Input):** your code-bundle dataset (`dist/sentinel_gate_code.zip`, Stage 7 version) and a dataset with `dist/sentinel_v1_Win64.zip`.

**Safety:** feature records only; no binaries are downloaded or executed; `EMBER2024-capa` is refused.

**Kaggle settings:** Accelerator **None** (CPU only), Internet **On**. Then **Save Version → Save & Run All (Commit)**.


In [ ]:
# 1) Settings + platform + disk/RAM check
FILE_TYPE = "Win64"                 # fixed by configs/experiment.yaml
POLICIES = ["last_n_weeks"]         # <-- run 1: last_n_weeks; run 2: all_history; run 3: recency_weighted
PART = "last_n_weeks"               # <-- names the output zip; use the policy name
REQUIRED_FREE_GB = 45

import os, shutil, subprocess, sys, zipfile, hashlib, glob, time, importlib.util, json
from pathlib import Path

assert all(p in ("all_history", "last_n_weeks", "recency_weighted") for p in POLICIES), POLICIES
ON_KAGGLE = ("KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
             or (Path("/kaggle/input").is_dir() and any(Path("/kaggle/input").iterdir())))
ON_COLAB = not ON_KAGGLE and ("COLAB_RELEASE_TAG" in os.environ
                              or importlib.util.find_spec("google.colab") is not None)
WHERE = "Kaggle notebook" if ON_KAGGLE else ("Google Colab" if ON_COLAB else "other notebook host")

# Never put bulk data in /kaggle/working: Kaggle saves that folder as notebook output (20 GB cap).
candidates = [Path(p) for p in ["/content", "/kaggle/tmp", "/tmp"] if Path(p).exists()]
free = {p: shutil.disk_usage(p).free / 1e9 for p in candidates}
SCRATCH = max(free, key=free.get)
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else SCRATCH / "sentinel_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f"Host: {WHERE} | policies: {POLICIES} | scratch: {SCRATCH} ({free[SCRATCH]:.1f} GB free) | "
      f"outputs: {OUT_DIR} | CPUs: {os.cpu_count()} | RAM: {ram_gb:.1f} GB")
if free[SCRATCH] < REQUIRED_FREE_GB:
    raise SystemExit(f"Not enough disk: need {REQUIRED_FREE_GB} GB free. Tell Claude.")
if ram_gb < 24:
    raise SystemExit(f"All-history retraining needs ~16 GB RAM; this machine has {ram_gb:.1f} GB. Tell Claude.")

CODE, DATA, VENV = SCRATCH / "sentinel_code", SCRATCH / "sentinel_data", SCRATCH / "sentinel_venv"
WORK = SCRATCH / "stage7_work"
V1_DIR = SCRATCH / "v1_bundle"
(DATA / "raw").mkdir(parents=True, exist_ok=True)


In [ ]:
# 2) Load the code bundle (Kaggle: attached dataset; Colab: upload when asked)
BUNDLE_NAME = "sentinel_gate_code.zip"

def find_bundle():
    zips = glob.glob(f"/kaggle/input/**/{BUNDLE_NAME}", recursive=True) + glob.glob(f"/content/{BUNDLE_NAME}")
    if zips:
        return Path(zips[0]), None
    dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/BUNDLE_INFO.json", recursive=True)]
    return None, (dirs[0] if dirs else None)

bundle, extracted = find_bundle()
if bundle is None and extracted is None and ON_COLAB:
    from google.colab import files
    files.upload()
    bundle, extracted = find_bundle()
if bundle is None and extracted is None:
    raise SystemExit(f"{BUNDLE_NAME} not found. Kaggle: Add Input -> your code-bundle dataset.")

shutil.rmtree(CODE, ignore_errors=True)
if bundle is not None:
    print(f"bundle: {bundle}\nsha256: {hashlib.sha256(bundle.read_bytes()).hexdigest()}")
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(CODE)
else:
    print(f"using extracted bundle at {extracted}")
    shutil.copytree(extracted, CODE)
for need in ("scripts/experiment_kaggle.py", "configs/experiment.yaml",
             "src/sentinel/experiment/export.py", f"docs/cleaning/{FILE_TYPE}.json"):
    if not (CODE / need).exists():
        raise SystemExit(f"Bundle has no {need}: it is an old bundle. Upload the Stage 7 dist/sentinel_gate_code.zip.")
print(open(CODE / "BUNDLE_INFO.json").read()[:200], "...")

In [ ]:
# 2b) The v1 bundle (uploaded dist/sentinel_v1_Win64.zip): allow-listed names only.
# Kaggle usually UNPACKS an uploaded zip into a folder, so accept either the zip itself or a
# folder holding exactly the 4 v1 files (the v1 run_info has "train_config"; candidates don't).
V1_NAME = f"sentinel_v1_{FILE_TYPE}.zip"
ALLOWED = {"lightgbm.txt", "lightgbm_calibration.npz", "metrics.json", "run_info.json"}
found = glob.glob(f"/kaggle/input/**/{V1_NAME}", recursive=True) + glob.glob(f"/content/{V1_NAME}")
folders = []
for ri in glob.glob("/kaggle/input/**/run_info.json", recursive=True):
    d = Path(ri).parent
    if ALLOWED <= {p.name for p in d.iterdir()} and "train_config" in json.loads(Path(ri).read_text()):
        folders.append(d)
if not found and not folders and ON_COLAB:
    from google.colab import files
    files.upload()
    found = glob.glob(f"/content/{V1_NAME}")
shutil.rmtree(V1_DIR, ignore_errors=True)
V1_DIR.mkdir(parents=True)
if found:
    with zipfile.ZipFile(found[0]) as zf:
        names = set(zf.namelist())
        if names != ALLOWED:
            raise SystemExit(f"unexpected files in {V1_NAME}: {sorted(names ^ ALLOWED)}")
        for n in sorted(ALLOWED):
            (V1_DIR / n).write_bytes(zf.read(n))
    print("v1 from zip:", found[0])
elif len(folders) == 1:
    for n in sorted(ALLOWED):
        shutil.copyfile(folders[0] / n, V1_DIR / n)
    print("v1 from extracted folder:", folders[0])
elif len(folders) > 1:
    raise SystemExit(f"several v1 bundles attached: {[str(f) for f in folders]}. Attach only one.")
else:
    raise SystemExit(f"{V1_NAME} not found (neither the zip nor its unpacked folder). "
                     f"Kaggle: Add Input -> the dataset holding dist/{V1_NAME}.")
for n in sorted(ALLOWED):    # compare with the sha256 printed by `make v1-zip`
    print(n, hashlib.sha256((V1_DIR / n).read_bytes()).hexdigest())


In [ ]:
# 3) Install locked dependencies into an isolated Python 3.11 venv
def run(cmd, env=None, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), env=env, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if '"event": "clean_step"' not in line:
            print(line, end="", flush=True)
    if proc.wait() != 0:
        raise SystemExit(f"command failed with exit code {proc.returncode}")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.13.6"])
env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(VENV), UV_PYTHON="3.11", UV_LINK_MODE="copy")
run([sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--no-install-project"], env=env, cwd=CODE)
PY = VENV / "bin" / "python"
RUN_ENV = dict(os.environ, PYTHONPATH=f"{CODE / 'src'}:{CODE}", SENTINEL_DATA_DIR=str(DATA),
               HF_HOME=str(DATA / ".hf-home"), PYTHONUNBUFFERED="1")
run([PY, "-c", "import sys, thrember, lightgbm, sklearn; print(sys.version.split()[0], 'lightgbm', lightgbm.__version__, 'sklearn', sklearn.__version__)"], env=RUN_ENV)

In [ ]:
# 4) Download -> rebuild splits -> check -> retrain candidates -> export scores
t0 = time.time()
run([PY, "scripts/download_ember.py", "--file-types", FILE_TYPE, "--dest", DATA / "raw"], env=RUN_ENV, cwd=CODE)
t1 = time.time()
run([PY, "scripts/clean_data.py", "--file-types", FILE_TYPE, "--raw-dir", DATA / "raw",
     "--splits-dir", DATA / "splits", "--clean-dir", DATA / "clean",
     "--manifest-dir", SCRATCH / "rebuilt_manifests", "--report", SCRATCH / "rebuilt_report.md",
     "--workers", str(os.cpu_count() or 1)], env=RUN_ENV, cwd=CODE)
for z in (DATA / "raw").glob(f"{FILE_TYPE}_*.zip"):
    z.unlink()                                  # free disk
shutil.rmtree(DATA / ".hf-home", ignore_errors=True)
t2 = time.time()

out_zip = OUT_DIR / f"sentinel_stage7b_{FILE_TYPE}_{PART}.zip"
run([PY, "scripts/experiment_kaggle.py", "--file-type", FILE_TYPE, "--splits-dir", DATA / "splits",
     "--clean-dir", DATA / "clean", "--v1-dir", V1_DIR, "--work", WORK, "--out-zip", out_zip,
     "--policies", *POLICIES, "--threads", str(os.cpu_count() or 0),
     "--label-delay-weeks", "6"], env=RUN_ENV, cwd=CODE)
t3 = time.time()
print(f"\n== download {(t1 - t0) / 60:.1f} min, clean {(t2 - t1) / 60:.1f} min, "
      f"candidates + export {(t3 - t2) / 60:.1f} min")
print("result:", out_zip, f"{out_zip.stat().st_size / 1e6:.1f} MB",
      "sha256", hashlib.sha256(out_zip.read_bytes()).hexdigest())
if ON_COLAB:
    from google.colab import files
    files.download(str(out_zip))


In [ ]:
# 5) Summary (download the zip from the Output tab and give its path to Claude)
with zipfile.ZipFile(out_zip) as zf:
    print(sorted(zf.namelist()))
    info = json.loads(zf.read("export_info.json"))
    models = json.loads(zf.read("models.json"))
print(f"rows scored: {info['rows']:,} (weeks {info['score_weeks'][0]}-{info['score_weeks'][-1]}), "
      f"rebuild check: {info['rebuild_check']['result']}")
for mid, m in models.items():
    print(f"{mid:24s} T={m['as_of_week']} threshold {m['threshold_raw']:.7f} fit rows {m['fit_rows']:,} "
          f"never-break flags {m['never_break_flags_at_training']}")
print("\nKaggle: download", out_zip.name, "from the Output tab.")
